# Crude oil Power BI project - data preparation

Downloads daily WTI crude (Yahoo Finance `CL=F`), cleans it, builds the calendar, regimes and events tables, and adds Brent, EUR/USD, DXY, DAX, XLE (Yahoo), US rates and CPI (FRED) and ECB rates and euro-area inflation.

**Output, ready to load into Power BI (folder `powerbi_data/`):**

| File | Power BI table | One row per |
|---|---|---|
| `fact_daily.csv` | `fact_daily` | WTI trading day |
| `dim_date.csv` | `dim_date` | calendar day |
| `dim_regime.csv` | `dim_regime` | market period (6) |
| `dim_event.csv` | `dim_event` | dated event (40) |

Cleaned intermediate tables go to `intermediate/` and are not loaded into Power BI.

**Frozen data:** `END_DATE` in the parameters cell sets the last day of data (28 Oct 2026, included), so a rerun never adds newer days. Set `EVENTS_ONLY = True` to rebuild only `dim_event.csv` after editing `events_template.csv`, without downloading anything.

**Run order:** 1 setup, 2 functions, 3 parameters, 4 run, 5 checks.

The market data are downloaded, not stored in the repository (Yahoo Finance data is for personal use). A full rerun downloads all history again, so past values can differ slightly if a source revised them.

## 1. Setup

In [ ]:
# Run once, then comment out
# %pip install pandas numpy requests yfinance matplotlib

In [ ]:
import io, sys
from pathlib import Path

import numpy as np
import pandas as pd

START = "2016-01-01"
OUT_DIR = Path("powerbi_data")        # the four tables Power BI loads
INTERIM_DIR = Path("intermediate")    # cleaned WTI, market and rates tables (not loaded into Power BI)

# ---- Regimes: analyst judgement. Edit the dates if you disagree. -------------
REGIMES = [
    ("Pre-COVID", "2016-01-01", "2020-02-19"),
    ("COVID shock", "2020-02-20", "2020-12-31"),
    ("Reopening recovery", "2021-01-01", "2022-02-23"),
    ("Ukraine war spike", "2022-02-24", "2022-12-31"),
    ("Post-spike normalisation", "2023-01-01", "2026-02-27"),
    ("Hormuz crisis", "2026-02-28", "2099-12-31"),
]

# ---- Yahoo Finance tickers (market context) ----------------------------------
YAHOO = {
    "BRENT": "BZ=F",
    "EURUSD": "EURUSD=X",
    "DXY": "DX-Y.NYB",
    "DAX": "^GDAXI",
    "XLE": "XLE",
}

# ---- FRED series (no API key needed for the CSV endpoint) --------------------
FRED_DAILY = {"FED_FUNDS": "DFF", "US10Y": "DGS10"}
FRED_MONTHLY = {"US_CPI_INDEX": "CPIAUCSL", "US_RECESSION": "USREC"}

# ---- ECB Data Portal (SDMX CSV) ----------------------------------------------
ECB = {
    # each entry: list of keys in order of preference; later keys only fill dates the earlier ones lack
    "ECB_DEPOSIT_RATE": ["FM/D.U2.EUR.4F.KR.DFR.LEV"],   # daily, changes only on decisions
    # Euro-area inflation, annual rate %. The old ICP dataset stops in Dec 2025; the new HICP dataset
    # (4D0 = new classification) carries 2026 onward, so splice the two.
    "EA_HICP_YOY": ["HICP/M.U2.N.000000.4D0.ANR", "ICP/M.U2.N.000000.4.ANR"],
}

# ---- Optional: weekly German fuel prices ---------------------------------------
# EU Weekly Oil Bulletin has a "price history" Excel file (search "Weekly Oil Bulletin"
# on energy.ec.europa.eu). Filter Germany and save a CSV with exactly these columns:
#   date,euro95_eur_per_1000l,diesel_eur_per_1000l
# Use prices WITH taxes for the consumer view. Put it at data_raw/fuel_germany_weekly.csv
FUEL_FILE = Path("data_raw/fuel_germany_weekly.csv")

BARREL_LITRES = 158.987

## 2. Functions
### WTI cleaning

In [ ]:
def clean_wti(path: str, keep_today: bool = False, drop_last_row: bool = False) -> pd.DataFrame:
    raw = pd.read_csv(path, na_values=["NA", ""], parse_dates=["Date"])
    raw.columns = [c.strip() for c in raw.columns]
    n0 = len(raw)

    df = raw.dropna(subset=["Close"]).copy()
    print(f"[WTI] {n0} rows; dropped {n0 - len(df)} all-NA holiday rows")

    # Adjusted_Close duplicates Close on a futures series; check before dropping it
    if "Adjusted_Close" in df.columns:
        same = np.allclose(df["Close"], df["Adjusted_Close"], equal_nan=True)
        print(f"[WTI] Adjusted_Close identical to Close: {same} -> dropped")
        df = df.drop(columns="Adjusted_Close")

    df = df.sort_values("Date").drop_duplicates("Date").reset_index(drop=True)

    # Frozen data window: ignore anything after END_DATE (END_DATE itself is kept)
    n_before = len(df)
    df = df[df["Date"] <= pd.Timestamp(END_DATE)].reset_index(drop=True)
    if len(df) < n_before:
        print(f"[WTI] dropped {n_before - len(df)} rows after END_DATE {END_DATE}")

    # The newest row may be a session still in progress (partial volume)
    today = pd.Timestamp.today().normalize()
    if not keep_today and df["Date"].iloc[-1] >= today:
        print(f"[WTI] dropped in-progress session {df['Date'].iloc[-1].date()} (use --keep-today to keep)")
        df = df.iloc[:-1].copy()

    if drop_last_row:   # e.g. the file was downloaded mid-session and today's date has passed since
        print(f"[WTI] dropped last row {df['Date'].iloc[-1].date()} (--drop-last-row)")
        df = df.iloc[:-1].copy()

    df = df.rename(columns={"Date": "date", "Open": "open", "High": "high",
                            "Low": "low", "Close": "close", "Volume": "volume"})

    # --- data-quality flags ---------------------------------------------------
    df["is_negative_price"] = (df[["open", "high", "low", "close"]] <= 0).any(axis=1)
    df["is_zero_volume"] = df["volume"] == 0
    df["is_repeated_volume"] = df["volume"].diff() == 0   # suspicious Yahoo repeats

    # --- returns, guarded against non-positive prices -----------------------------
    prev = df["close"].shift(1)
    ok = (df["close"] > 0) & (prev > 0)
    df["abs_change"] = df["close"] - prev                               # USD/bbl, always defined
    df["pct_change"] = np.where(ok, df["close"] / prev - 1, np.nan)     # NaN around the 2020 negative price
    df["log_return"] = np.log(np.where(ok, df["close"] / prev, np.nan))
    df["close_positive"] = df["close"].where(df["close"] > 0)           # use for ratio-based DAX

    df["date"] = df["date"].dt.date
    return df

### Downloads (Yahoo, FRED, ECB)

In [ ]:
def download_wti(path: str, ticker: str = "CL=F") -> pd.DataFrame:
    """Download daily WTI front-month futures from Yahoo Finance and save a CSV with the same
    columns as the R export: Date, Open, High, Low, Close, Volume, Adjusted_Close."""
    import yfinance as yf
    end = str((pd.Timestamp(END_DATE) + pd.Timedelta(days=1)).date())   # yfinance end is exclusive, so END_DATE itself is included
    d = yf.download(ticker, start=START, end=end, auto_adjust=False,
                    progress=False, multi_level_index=False)
    if d.empty:
        raise RuntimeError(f"Yahoo returned no data for {ticker}")
    d = d.reset_index().rename(columns={"Adj Close": "Adjusted_Close"})
    d["Date"] = pd.to_datetime(d["Date"]).dt.tz_localize(None).dt.strftime("%Y-%m-%d")
    d = d[["Date", "Open", "High", "Low", "Close", "Volume", "Adjusted_Close"]]
    d.to_csv(path, index=False)
    print(f"[WTI download] {ticker}: {len(d)} rows, {d['Date'].iloc[0]} to {d['Date'].iloc[-1]} -> saved {path}")
    return d


def fetch_yahoo(end: str) -> pd.DataFrame:
    import yfinance as yf
    frames = []
    for name, ticker in YAHOO.items():
        d = yf.download(ticker, start=START, end=end, auto_adjust=False,
                        progress=False, multi_level_index=False)
        if d.empty:
            print(f"[Yahoo] WARNING: no data for {ticker}")
            continue
        d = d.reset_index()[["Date", "Close"]].rename(columns={"Date": "date", "Close": name.lower()})
        d["date"] = pd.to_datetime(d["date"]).dt.tz_localize(None)
        frames.append(d.set_index("date"))
        print(f"[Yahoo] {ticker}: {len(d)} rows")
    return pd.concat(frames, axis=1, sort=True).reset_index() if frames else pd.DataFrame()


def fetch_fred(series_id: str) -> pd.Series:
    import requests
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd=2015-01-01"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    d = pd.read_csv(io.StringIO(r.text), na_values=["."])
    d.columns = ["date", "value"]            # first column is DATE or observation_date
    d["date"] = pd.to_datetime(d["date"])
    print(f"[FRED] {series_id}: {len(d)} rows, {d['date'].min().date()} to {d['date'].max().date()}")
    return d.set_index("date")["value"].astype(float)


def fetch_ecb(key: str) -> pd.Series:
    import requests
    url = f"https://data-api.ecb.europa.eu/service/data/{key}?format=csvdata&startPeriod=2015-01-01"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    d = pd.read_csv(io.StringIO(r.text))
    d["date"] = pd.to_datetime(d["TIME_PERIOD"])      # monthly 'YYYY-MM' parses to the 1st
    print(f"[ECB] {key}: {len(d)} rows, {d['date'].min().date()} to {d['date'].max().date()}")
    return d.set_index("date")["OBS_VALUE"].astype(float)

### Build tables

In [ ]:
def build_dim_date(first: str, last: str) -> pd.DataFrame:
    d = pd.DataFrame({"date": pd.date_range(first, last, freq="D")})
    d["year"] = d.date.dt.year
    d["quarter"] = "Q" + d.date.dt.quarter.astype(str)
    d["month_num"] = d.date.dt.month
    d["month_name"] = d.date.dt.strftime("%b")
    d["year_month"] = d.date.dt.strftime("%Y-%m")
    d["iso_week"] = d.date.dt.isocalendar().week.astype(int)
    d["weekday_num"] = d.date.dt.weekday + 1
    d["is_weekday"] = d.weekday_num <= 5
    d["regime"] = None
    for name, a, b in REGIMES:
        d.loc[(d.date >= a) & (d.date <= b), "regime"] = name
    d["date"] = d.date.dt.date
    return d


def build_rates(cal: pd.DatetimeIndex) -> pd.DataFrame:
    """Daily rates table. Monthly/irregular series are forward-filled (step function)."""
    out = pd.DataFrame(index=cal)
    for name, sid in FRED_DAILY.items():
        s = fetch_fred(sid)
        out[name.lower()] = s.reindex(cal.union(s.index)).ffill().reindex(cal)
    cpi = fetch_fred(FRED_MONTHLY["US_CPI_INDEX"])
    cpi_yoy = (cpi.pct_change(12) * 100).dropna()
    out["us_cpi_yoy"] = cpi_yoy.reindex(cal.union(cpi_yoy.index)).ffill().reindex(cal)
    rec = fetch_fred(FRED_MONTHLY["US_RECESSION"])
    out["us_recession"] = rec.reindex(cal.union(rec.index)).ffill().reindex(cal)
    for name, keys in ECB.items():
        s = None
        for key in keys:
            try:
                part = fetch_ecb(key)
            except Exception as e:
                print(f"[ECB] {key} failed ({e}); trying next key")
                continue
            s = part if s is None else s.combine_first(part)   # earlier keys win on overlapping dates
        if s is None:
            print(f"[ECB] WARNING: no data for {name}")
            continue
        print(f"[ECB] {name} combined: {s.index.min().date()} to {s.index.max().date()}, {len(s)} points")
        out[name.lower()] = s.reindex(cal.union(s.index)).ffill().reindex(cal)
    out.index.name = "date"
    out = out.reset_index()
    out["date"] = out["date"].dt.date
    return out


def build_fact_daily(wti: pd.DataFrame, market: pd.DataFrame, rates: pd.DataFrame, max_fill_days: int = 3) -> pd.DataFrame:
    """One row per WTI trading day: WTI + Brent/FX/equities + rates, plus a trading-day counter.
    Market gaps (other markets closed that day) are filled forward by at most `max_fill_days` rows.
    `n_market_fields_filled` counts the filled market fields of the day; a filled field gets no
    return that day (`<field>_log_return` is blank), so a holiday never shows up as a 0% move."""
    spine = pd.to_datetime(wti["date"])
    mk = market.copy()
    mk["date"] = pd.to_datetime(mk["date"])
    mk = mk.set_index("date").sort_index()
    cols = ["brent", "eurusd", "dxy", "dax", "xle"]
    mk = mk[cols]
    on_spine = mk.reindex(spine)                       # raw values on WTI trading days
    filled = mk.ffill(limit=max_fill_days).reindex(spine)
    was_filled = on_spine.isna() & filled.notna()

    out = wti.copy()
    out["date"] = spine.values
    out = out.set_index("date")
    out.insert(0, "trading_day_seq", np.arange(1, len(out) + 1))
    out = out.join(filled)
    for c in cols:
        s = filled[c].where(filled[c] > 0)
        r = np.log(s / s.shift(1))
        r[was_filled[c]] = np.nan
        out[f"{c}_log_return"] = r.values
    out["n_market_fields_filled"] = was_filled.sum(axis=1).astype(int).values

    rt = rates.copy()
    rt["date"] = pd.to_datetime(rt["date"])
    out = out.join(rt.set_index("date"))
    out = out.reset_index()
    out["date"] = out["date"].dt.date
    return out


def add_event_anchors(events: pd.DataFrame, fact_daily: pd.DataFrame) -> pd.DataFrame:
    """Map each event to the first trading day on or after its date (weekend events -> Monday)."""
    days = pd.to_datetime(fact_daily["date"]).to_numpy()
    seq = fact_daily["trading_day_seq"].to_numpy()
    idx = np.searchsorted(days, pd.to_datetime(events["event_date"]).to_numpy(), side="left")
    ok = idx < len(days)
    ev = events.copy()
    ev["anchor_date"] = [pd.Timestamp(days[i]).date() if o else None for i, o in zip(idx, ok)]
    ev["anchor_seq"] = pd.array([int(seq[i]) if o else None for i, o in zip(idx, ok)], dtype="Int64")
    ev["is_day_precision"] = ev["date_precision"].eq("day")
    return ev


def passthrough_analysis(wti: pd.DataFrame, market: pd.DataFrame) -> None:
    """Weekly lead/lag correlation: WTI in EUR/litre vs German pump prices."""
    if not FUEL_FILE.exists():
        print(f"[Fuel] {FUEL_FILE} not found - skipping pass-through analysis")
        return
    fuel = pd.read_csv(FUEL_FILE, parse_dates=["date"])
    m = (wti[["date", "close_positive"]].assign(date=lambda x: pd.to_datetime(x["date"]))
         .merge(market[["date", "eurusd"]].assign(date=lambda x: pd.to_datetime(x["date"])),
                on="date", how="inner"))
    m["wti_eur_l"] = m["close_positive"] / m["eurusd"] / BARREL_LITRES
    wk = m.set_index("date")["wti_eur_l"].resample("W-MON").last().rename("wti_eur_l")
    f = fuel.set_index("date").resample("W-MON").last()
    j = pd.concat([wk, f], axis=1).dropna()
    for c in ["wti_eur_l", "euro95_eur_per_1000l", "diesel_eur_per_1000l"]:
        j[f"d_{c}"] = np.log(j[c]).diff()
    rows = []
    for fuel_col in ["euro95_eur_per_1000l", "diesel_eur_per_1000l"]:
        for lag in range(0, 9):
            rows.append({"fuel": fuel_col.split("_")[0], "lag_weeks": lag,
                         "corr_weekly_change": j["d_wti_eur_l"].shift(lag).corr(j[f"d_{fuel_col}"])})
    pd.DataFrame(rows).to_csv(OUT_DIR / "passthrough_lags.csv", index=False)
    j.reset_index().to_csv(OUT_DIR / "fact_fuel_weekly.csv", index=False)
    print("[Fuel] wrote fact_fuel_weekly.csv and passthrough_lags.csv")

## 3. Parameters

In [ ]:
END_DATE = "2026-10-28"                 # last day of data (included). The dataset is frozen here, so reruns never add newer days
WTI_PATH = "crude_oil_2016_2026.csv"    # WTI file: downloaded below, or your existing file
EVENTS_PATH = "events_template.csv"     # keep this file next to the notebook
DOWNLOAD_WTI = True                     # True = download CL=F from Yahoo Finance; False = use the existing WTI_PATH file
SKIP_FETCH = False                      # True = no downloads for market/rates data; reads the files saved in intermediate/ by an earlier run
EVENTS_ONLY = False                     # True = only rebuild dim_event.csv from events_template.csv (no downloads, nothing else changes)
DROP_LAST_ROW = False                   # True only if your newest row is a half-finished session
KEEP_TODAY = False                      # today's partial session is dropped automatically


## 4. Run

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
INTERIM_DIR.mkdir(exist_ok=True)


def parse_event_dates(col: pd.Series, ids: pd.Series) -> pd.Series:
    """Parse event_date. Accepts 2020-03-08 (ISO), 08.03.2020 and 8/3/2020 (day first, as Excel
    often writes them in a German or European setting). Stops with a list of rows it cannot read."""
    s = col.astype(str).str.strip().str[:10] if col.dtype == object else col.astype(str)
    out = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    n_iso = int(out.notna().sum())
    for fmt in ("%d.%m.%Y", "%d/%m/%Y"):
        miss = out.isna()
        out[miss] = pd.to_datetime(col.astype(str).str.strip()[miss], format=fmt, errors="coerce")
    bad = out.isna()
    if bad.any():
        rows = pd.DataFrame({"event_id": ids[bad], "event_date_in_file": col[bad]})
        raise ValueError("These event_date values are not valid dates (use YYYY-MM-DD, e.g. 2020-03-08):\n"
                         + rows.to_string(index=False))
    if n_iso < len(out):
        print(f"[Events] {len(out) - n_iso} dates were not in YYYY-MM-DD form and were read day first (dd.mm.yyyy).")
    return out


def read_events() -> pd.DataFrame:
    """Read events_template.csv. Excel often saves CSV files in Windows-1252 (ANSI) instead of UTF-8,
    so try UTF-8 first and fall back to Windows-1252 (en dashes, accents and quotes then read correctly)."""
    try:
        ev = pd.read_csv(EVENTS_PATH, encoding="utf-8-sig", sep=None, engine="python")
    except UnicodeDecodeError:
        ev = pd.read_csv(EVENTS_PATH, encoding="cp1252", sep=None, engine="python")
        print("[Events] file is not UTF-8; read as Windows-1252. The output file dim_event.csv is written as UTF-8.")
    ev.columns = [c.strip() for c in ev.columns]
    ev = ev.dropna(how="all").reset_index(drop=True)          # drop empty rows Excel may add
    ev["event_date"] = parse_event_dates(ev["event_date"], ev["event_id"]).dt.date
    return ev


events = read_events()
print(f"[Events] {len(events)} events\n{events['verification'].value_counts().to_string()}")

if not EVENTS_ONLY:
    # --- WTI -----------------------------------------------------------------------
    if DOWNLOAD_WTI:
        download_wti(WTI_PATH)
    wti = clean_wti(WTI_PATH, keep_today=KEEP_TODAY, drop_last_row=DROP_LAST_ROW)
    wti.to_csv(INTERIM_DIR / "fact_wti_daily.csv", index=False)
    last = pd.to_datetime(wti["date"]).max()

    # --- calendar and regimes ----------------------------------------------------------
    dim_date = build_dim_date(START, str(last.date()))
    dim_regime = pd.DataFrame(REGIMES, columns=["regime", "start_date", "end_date"])

    # --- market, rates ------------------------------------------------------------------
    if SKIP_FETCH:
        market = pd.read_csv(INTERIM_DIR / "fact_market_daily.csv")
        rates = pd.read_csv(INTERIM_DIR / "fact_rates_daily.csv")
        print("[Offline] market and rates read from intermediate/")
    else:
        cal = pd.date_range(START, last, freq="D")
        market = fetch_yahoo(str((last + pd.Timedelta(days=1)).date()))
        market["date"] = pd.to_datetime(market["date"]).dt.date
        market.to_csv(INTERIM_DIR / "fact_market_daily.csv", index=False)
        rates = build_rates(cal)
        rates.to_csv(INTERIM_DIR / "fact_rates_daily.csv", index=False)
        passthrough_analysis(wti, market)   # only runs if data_raw/fuel_germany_weekly.csv exists


### Build and write the Power BI tables
`fact_daily` has one row per oil trading day with WTI, Brent, EUR/USD, DXY, DAX, XLE and the rates. Gaps from other markets' holidays are filled forward by at most 3 days (`n_market_fields_filled` shows where). `dim_event` gets the trading day each event maps to (`anchor_date`, `anchor_seq`). With `EVENTS_ONLY = True` only `dim_event.csv` is rewritten, from the existing `fact_daily.csv`.

In [ ]:
if EVENTS_ONLY:
    fact_daily = pd.read_csv(OUT_DIR / "fact_daily.csv")
else:
    fact_daily = build_fact_daily(wti, market, rates)
    fact_daily.to_csv(OUT_DIR / "fact_daily.csv", index=False)
    dim_date.to_csv(OUT_DIR / "dim_date.csv", index=False)
    dim_regime.to_csv(OUT_DIR / "dim_regime.csv", index=False)

dim_event = add_event_anchors(events, fact_daily)
dim_event.to_csv(OUT_DIR / "dim_event.csv", index=False)

print(f"fact_daily.csv: {len(fact_daily)} rows, {fact_daily.shape[1]} columns, "
      f"{fact_daily['date'].min()} to {fact_daily['date'].max()}")
print(f"dim_event.csv: {len(dim_event)} events, {int(dim_event['anchor_seq'].notna().sum())} anchored to a trading day")


## 5. Checks

In [ ]:
# Final checks on the files Power BI will load
tables = {name: pd.read_csv(OUT_DIR / f"{name}.csv") for name in ["fact_daily", "dim_date", "dim_regime", "dim_event"]}
for name, t in tables.items():
    print(f"{name}.csv: {len(t)} rows, {t.shape[1]} columns")

fd, de = tables["fact_daily"], tables["dim_event"]
assert fd["date"].is_unique, "fact_daily has duplicate dates"
assert (fd["trading_day_seq"] == range(1, len(fd) + 1)).all(), "trading_day_seq is not 1..n"
assert de["anchor_seq"].notna().all(), "some events are not anchored to a trading day"
assert de["anchor_seq"].max() <= fd["trading_day_seq"].max()
print(f"Negative-price days: {int(fd['is_negative_price'].sum())} (expected 2, April 2020)")
print(f"Last day in fact_daily: {fd['date'].iloc[-1]}")
if fd["date"].iloc[-1] != END_DATE:
    print(f"WARNING: last day is not END_DATE ({END_DATE}). Check the Yahoo download for that day.")
else:
    print(f"OK: data end on END_DATE ({END_DATE}), which is included.")
print("\nVerification status:\n" + de["verification"].value_counts().to_string())

### WTI checks (skipped when `EVENTS_ONLY = True`)

In [ ]:
if not EVENTS_ONLY:
    display(wti.tail())
    display(wti.loc[wti["date"].astype(str).between("2020-04-17", "2020-04-23"),
                    ["date", "close", "abs_change", "pct_change", "log_return", "is_negative_price"]])
    print("Zero-volume rows:", int(wti.is_zero_volume.sum()), "| repeated-volume rows:", int(wti.is_repeated_volume.sum()))

In [ ]:
if not EVENTS_ONLY:
    import matplotlib.pyplot as plt
    w = wti.assign(date=pd.to_datetime(wti["date"]))
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(w["date"], w["close"], lw=1)
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_title("WTI front-month close (USD/bbl)")
    plt.show()